last modified date : 2026.07   
제작 : 모두의연구소 퍼실팀

# Day 2 최종 제출 프로젝트  
## Advanced·Modular RAG 구현과 RAGAS 기반 비교 평가

이 노트북은 KorQuAD v1과 KLUE-MRC를 이용해 기본 Naive RAG를 구축하고,  
Multi-Query, RAG-Fusion, HyDE, Cross-Encoder Reranking, 간소화 Self-RAG를 단계적으로 구현한다.

정량 비교에 사용하는 **Advanced RAG**는 다음 구조로 고정하였다.

```text
Dense Retrieval top-10
→ Multilingual Cross-Encoder Reranking
→ 상위 top-3 문서 선택
→ 근거 기반 답변 생성
```

Multi-Query, RAG-Fusion, HyDE, Self-RAG는 각각 독립적인 확장 기법으로 구현하며,  
정량 비교용 Advanced 체인에 모두 섞지 않아 각 기법의 역할을 명확하게 구분한다.


# 들어가며

대규모 언어 모델은 자연스러운 문장을 생성할 수 있지만, 학습 이후의 최신 정보나 특정 조직의 내부 문서를 자동으로 알 수는 없다. 또한 근거가 부족한 상황에서도 그럴듯한 내용을 만들어 내는 환각 문제가 발생할 수 있다.

RAG(Retrieval-Augmented Generation)는 질문과 관련된 외부 문서를 먼저 검색하고, 검색된 문서를 근거로 답변을 생성한다. 그러나 단순한 Naive RAG는 관련 없는 문서를 검색하거나 중요한 근거를 놓칠 수 있다. 본 프로젝트에서는 검색 전·후 단계의 여러 개선 기법을 구현하고, RAGAS의 네 가지 지표를 이용해 Naive RAG와 Advanced RAG를 비교한다.

### 프로젝트 목표

1. KorQuAD v1 기반 Naive RAG 베이스라인 구축
2. Multi-Query, RAG-Fusion, HyDE 검색 방식 구현
3. 다국어 Cross-Encoder Reranking 적용
4. 검색 필요성 판단과 근거 비평을 포함한 간소화 Self-RAG 구현
5. RAGAS를 이용한 Naive/Advanced 정량 비교
6. KLUE-MRC 뉴스 데이터로 도메인 변화에 따른 결과 분석
7. 실행 비용과 오류 가능성을 줄인 Colab 재현 환경 구성


## Step 0 : 설치와 실행 준비

### 권장 실행 순서

1. 아래 설치 셀을 한 번 실행한다.
2. 설치가 끝나면 **런타임 → 세션 다시 시작**을 누른다.
3. 다시 이 노트북의 Step 0 설정 셀부터 아래 방향으로 한 번씩 실행한다.
4. 기본값인 `QUICK_MODE=True`를 유지하면 API 호출과 평가 시간을 줄일 수 있다.
5. 50문항 통계 검정은 비용이 크므로 `RUN_BIG_STAT_TEST=False`를 유지한다.

> 같은 셀을 반복 실행해 결과가 누적되지 않도록 주요 리스트와 Chroma 컬렉션은 실행 시 초기화되도록 구성하였다.


In [1]:
# Colab pre-installed langchain 0.3 / ragas 0.1~0.4 를 ragas 0.2.10 호환 조합으로 정리합니다.
# 처음 실행 시 약 3~5분 걸립니다. 진행률 출력을 보면서 기다리세요 (멈춘 게 아닙니다).

# 1) 기존 langchain / ragas 패키지 제거 — 버전 충돌로 인한 pip resolver 백트래킹 방지
!pip uninstall -y ragas ragas-experimental langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-chroma

# 2) 0.2 시리즈 패치 버전까지 핀 설치 — resolver 부담 최소화 (-q 제거해서 진행률 보이게)
!pip install --no-cache-dir \
    "ragas==0.2.10" \
    "langchain==0.2.17" \
    "langchain-core==0.2.43" \
    "langchain-community==0.2.19" \
    "langchain-openai==0.1.25" \
    "langchain-text-splitters==0.2.4" \
    "langchain-chroma==0.1.4" \
    pypdf chromadb tiktoken sentence-transformers datasets nest_asyncio pandas

Found existing installation: ragas 0.2.10
Uninstalling ragas-0.2.10:
  Successfully uninstalled ragas-0.2.10
Found existing installation: langchain 0.2.17
Uninstalling langchain-0.2.17:
  Successfully uninstalled langchain-0.2.17
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-community 0.2.19
Uninstalling langchain-community-0.2.19:
  Successfully uninstalled langchain-community-0.2.19
Found existing installation: langchain-openai 0.1.25
Uninstalling langchain-openai-0.1.25:
  Successfully uninstalled langchain-openai-0.1.25
Found existing installation: langchain-text-splitters 0.2.4
Uninstalling langchain-text-splitters-0.2.4:
  Successfully uninstalled langchain-text-splitters-0.2.4
Found existing installation: langchain-chroma 0.1.4
Uninstalling langchain-chroma-0.1.4:
  Successfully uninstalled langchain-chroma-0.1.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

> ⚠️ **위 설치 셀(Step 0)을 실행한 뒤 반드시 [런타임 > 세션 다시 시작 (Restart session)]을 한 번 눌러주세요.**
>
> 이 셀은 Colab에 기본 설치된 langchain을 제거하고 `0.2.x` / `ragas 0.2.10` 조합으로 다운그레이드합니다. 이미 메모리에 로드된 패키지를 교체하는 것이라 Colab이 재시작을 요구합니다.
>
> 재시작 후에는 **설치 셀은 다시 실행하지 말고** 이 셀 아래(키 설정)부터 순서대로 실행하면 됩니다.

In [2]:
import os
import json
from pathlib import Path

# Chroma 익명 telemetry 비활성화
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import nest_asyncio
nest_asyncio.apply()

# ============================================================
# 실행 설정: 코랩 사용량이 적도록 기본값을 안전하게 구성했습니다.
# ============================================================
QUICK_MODE = True
EVAL_N = 8 if QUICK_MODE else 20
EVAL_N_KLUE = 8 if QUICK_MODE else 20
RUN_KORQUAD_RAGAS = True
RUN_KLUE_RAGAS = True
RUN_BIG_STAT_TEST = False       # 50문항 평가는 비용이 크므로 기본 비활성화
USE_HYDE_FALLBACK = True

print("실행 모드:", "QUICK" if QUICK_MODE else "FULL")
print(f"KorQuAD 평가 문항: {EVAL_N}개 / KLUE 평가 문항: {EVAL_N_KLUE}개")


실행 모드: QUICK
KorQuAD 평가 문항: 8개 / KLUE 평가 문항: 8개


In [3]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "Colab 왼쪽의 키 아이콘(Secrets)에서 OPENAI_API_KEY를 등록한 뒤 다시 실행하세요."
    )
os.environ["OPENAI_API_KEY"] = api_key
print("OPENAI_API_KEY 연결 완료")

OPENAI_API_KEY 연결 완료


## Step 1 : KorQuAD v1 위에서 Naive RAG 베이스라인 만들기

Day 1에서 만든 RAG 파이프라인을 한국어 QA 벤치마크 **KorQuAD v1** 위에 다시 한 번 올립니다. 이후 단계는 모두 이 베이스라인 위에 ‘덧붙이는’ 방식입니다.

- HuggingFace `datasets` 로 KorQuAD v1 자동 다운로드 (별도 PDF 업로드 불필요)
- 일부만 샘플링해 토큰 비용 통제 (unique context 약 200개)
- Embedding → VectorStore → Retriever → LLM
- 검색 전략은 단순 `similarity` (top-k)

**📥 데이터셋**: <https://huggingface.co/datasets/KorQuAD/squad_kor_v1>

In [4]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken

# API 키 안전 확인
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY가 없습니다. 바로 위 API 키 셀을 먼저 실행하세요.")

tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    return len(tokenizer.encode(text))

# QUICK_MODE에서는 인덱싱 문서 수를 줄여 비용과 시간을 절약합니다.
KORQUAD_SAMPLE_N = 800 if QUICK_MODE else 2000
raw_ds = (
    load_dataset("squad_kor_v1", split="validation")
    .shuffle(seed=42)
    .select(range(KORQUAD_SAMPLE_N))
)

unique = {}
for ex in raw_ds:
    unique.setdefault(ex["context"], ex["title"])
context_docs = [
    Document(page_content=context, metadata={"title": title})
    for context, title in unique.items()
]

# 문장 경계가 잘릴 때 문맥 손실을 줄이기 위해 50토큰 overlap 사용
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=tiktoken_len,
)
docs = splitter.split_documents(context_docs)

embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# 중복 실행 시 기존 메모리 DB와 섞이지 않도록 collection_name을 고정하고 초기화
try:
    if "db" in globals():
        db.delete_collection()
except Exception:
    pass

db = Chroma(
    collection_name="korquad_rag_final",
    embedding_function=embedding,
)
BATCH = 100
for i in range(0, len(docs), BATCH):
    db.add_documents(docs[i:i+BATCH])

naive_retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"베이스라인 준비 완료 — samples: {len(raw_ds)}, unique contexts: {len(context_docs)}, chunks: {len(docs)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


베이스라인 준비 완료 — samples: 800, unique contexts: 548, chunks: 836


베이스라인 RAG로 간단한 질의를 던져 답이 나오는지 확인합니다.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    """
다음 문서만 근거로 질문에 한국어로 답하세요.

규칙:
1. 문서에 실제로 있는 정보만 사용하세요.
2. 질문이 요구하는 대상과 단위를 정확히 확인하세요.
3. 답은 가능한 한 짧고 직접적으로 작성하세요.
4. 문서에서 확인할 수 없으면 '문서에서 확인할 수 없습니다.'라고 답하세요.

[문서]
{context}

[질문]
{question}

[답변]
"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("정답:", raw_ds[0]["answers"]["text"][0])
print("Naive RAG:", naive_chain.invoke(TEST_Q))


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
정답: 대중교통체계
Naive RAG: 대중교통체계입니다.


## Step 2 : Pre-retrieval 강화 — Multi-Query Retrieval  

사용자가 던진 질문 하나로만 검색하면 ‘다른 표현’으로 적힌 정답을 놓칠 수 있습니다. **Multi-Query Retrieval**은 LLM에게 ‘같은 의도의 다른 질문 N개’를 만들게 시킨 뒤, 각 질문으로 병렬 검색하고 결과를 합칩니다.

LangChain은 이를 한 클래스로 제공합니다.

In [6]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# 어떤 ‘유사 질문’으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['2004년 이명박 서울시장이 재직할 때 어떤 주요 개선 사항이 있었나요?  ', '이명박이 2004년에 서울시장으로서 추진한 주요 정책이나 변화는 무엇인가요?  ', '2004년 이명박 서울시장 재임 중에 이루어진 전반적인 개선 내용은 어떤 것들이 있나요?']


검색된 문서 수: 5
---
2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재선에 도전했다. 6월 2일에 치뤄진 지방선거에서 개표 초반에 한명숙 후보에게 뒤지다가, 후반 강남 3구의 개표가 시작되면서 역전하여 민선 5기 제34대 서울특별시장으로 재선되었다. 구체적으로 강남구(+59,206, +25.68%), 서초구(+43,820, +23.66%), 송파구(+23,814, +8.19%), 강동구(+11,097, +5.33%), 용산구(+8,579, +8.24%), 양천구(+1,078, +0.51%), 영


## Step 2.5 : RAG-Fusion — Multi-Query + RRF로 묶어내기

Day2_1 노트에서 “꼭 짚고 가라”고 했던 패턴 중 하나가 **RAG-Fusion** 입니다. Step 2의 Multi-Query는 ‘유사 질문 N개로 병렬 검색’ 까지만 했는데, **RAG-Fusion** 은 그 N개 검색 결과를 **Reciprocal Rank Fusion (RRF)** 라는 간단한 공식으로 합쳐 ‘여러 쿼리에서 공통으로 상위에 떴던 문서’ 를 최상단으로 끌어올립니다.

RRF 점수 공식:

$$
\text{score}(d) = \sum_{i=1}^{N} \frac{1}{k + \text{rank}_i(d)}
$$

- $\text{rank}_i(d)$ : i번째 쿼리의 결과에서 문서 $d$ 의 순위 (1부터)
- $k$ : 스무딩 상수 (관례적으로 60)

아래 셀에서는 (1) sub-query 생성, (2) 각 sub-query 로 검색, (3) **RRF 함수는 여러분이 직접 채우기**, (4) 결과 확인까지 한 번에 해봅니다.

In [7]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query 가 내부적으로 하는 일을 명시적으로 노출 (한국어)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 4개의 한국어 검색 쿼리를 만드세요. "
    "오직 4개의 쿼리만 한 줄에 하나씩 출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]


# (2) RRF 함수
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    """
    results_per_query : List[List[Document]]  쿼리별 검색 결과(순위 순).
    k                 : RRF smoothing 상수 (관례적으로 60).
    top_k             : 최종 반환할 문서 개수.
    """
    scores = defaultdict(float)
    docs_by_key = {}

    for docs in results_per_query:
        for rank, doc in enumerate(docs):  # rank 는 0부터
            key = doc.page_content
            scores[key] += 1.0 / (k + rank + 1)
            docs_by_key[key] = doc

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [docs_by_key[key] for key, _ in ranked[:top_k]]


# (3) 한 번 돌려보기
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for q in sub_queries:
    print(" -", q)

results_per_q = [db.similarity_search(q, k=5) for q in sub_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300] if fused else "(아직 TODO 가 비어 있어 결과가 없습니다)")

확장 질문 4개:
 - 2004년 이명박 서울시장 재직 중 개선한 사항은?
 - 이명박이 2004년 서울시장으로서 개선한 내용은 무엇인가?
 - 2004년 서울시장 이명박이 전면적으로 개선한 것은 어떤 것인가?
 - 이명박 서울시장 시절 2004년에 개선된 것은 무엇인가?

RAG-Fusion top-1 문서:
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교


## Step 3 : 패턴 ② HyDE — 가상의 ‘정답’으로 진짜 정답 찾기  

질문은 짧은 의문문, 정답은 긴 평서문이라 둘의 임베딩이 의외로 멀 수 있습니다. **HyDE(Hypothetical Document Embeddings)** 는 검색 전에 LLM에게 ‘가상의 정답’을 쓰게 한 뒤, 그 가상 답변을 임베딩해서 검색합니다.

직접 구현해 보겠습니다.

In [8]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    """
다음 질문과 관련된 문서에서 등장할 법한 핵심 개념과 표현을 포함한
검색용 가상 문단을 한국어로 작성하세요.

중요 규칙:
- 이 문단은 최종 답변이 아니라 검색용입니다.
- 확실하지 않은 고유명사, 날짜, 금액, 개수, 비율을 임의로 만들지 마세요.
- 질문의 핵심 용어와 가능한 동의 표현을 자연스럽게 포함하세요.

질문: {question}

검색용 가상 문단:
"""
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def hyde_retrieve(question, k=3):
    hypothetical = hyde_generator.invoke({"question": question})
    docs = db.similarity_search(hypothetical, k=k)
    return docs, hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 문단(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200] if docs_hyde else "검색 결과 없음")


가상 문단(HyDE):
 2004년 이명박 서울시장이 재직하던 시절, 서울시는 여러 가지 주요 개선 프로젝트를 추진하였습니다. 특히, 교통 체계의 효율성을 높이기 위한 대중교통 개선과 도로 확장 사업이 두드러졌습니다. 이명박 시장은 서울의 교통 혼잡 문제를 해결하기 위해 버스 전용 차선과 지하철 노선 확장을 적극적으로 시행하였으며, 이를 통해 시민들의 이동 편의성을 크게 향상시켰습니다. 또한, 환경 개선을 위한 녹지 공간 확대와 한강변 정비 사업도 중요한 정책으로 자리 잡았습니다. 이러한 변화들은 서울시의 도시 발전과 주민 생활 향상에 기여하였으며, 이명박 
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 4 : Post-retrieval 강화 — Cross-Encoder Reranking (multilingual)

검색 결과를 그대로 LLM 에 넘기지 않고, **Cross-encoder reranker** 가 (질문, 문단)을 함께 보면서 진짜 관련도를 다시 점수화합니다. 정밀도가 15~30% 개선되는 게 일반적인 보고입니다.

한국어 문서를 다루고 있으므로 다국어를 지원하는 cross-encoder 를 사용합니다. `BAAI/bge-reranker-v2-m3` 는 한국어를 포함한 100개 이상 언어에서 동작합니다. 처음 실행 시 모델 다운로드(~2GB)가 발생합니다.

In [9]:
from sentence_transformers import CrossEncoder

# 한국어를 포함하는 다국어 reranker. 첫 실행 시 모델 다운로드가 발생합니다.
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, docs, top_k=3):
    if not docs:
        return []
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs, show_progress_bar=False)
    ranked = sorted(zip(docs, scores), key=lambda x: float(x[1]), reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 상위 {len(top3)}개 선별")
print("최상위 문서:", top3[0].page_content[:200] if top3 else "검색 결과 없음")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

후보 10개 → Reranker 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 5 : Advanced RAG 체인 조립  

위에서 만든 컴포넌트들을 하나의 체인으로 묶습니다. **‘넓게 검색 → Reranker로 좁히기 → LLM 답변’** 패턴이 가장 흔히 쓰입니다.

In [10]:
def advanced_rag(question, candidate_k=10, final_k=3):
    """Dense 검색 후보를 넓게 확보한 뒤 Cross-Encoder로 재정렬하는 Advanced RAG."""
    candidates = db.as_retriever(search_kwargs={"k": candidate_k}).invoke(question)
    top_docs = rerank(question, candidates, top_k=final_k)

    if not top_docs:
        return "문서에서 확인할 수 없습니다.", []

    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": format_docs(top_docs),
        "question": question,
    })
    return answer, top_docs

ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("정답:", raw_ds[0]["answers"]["text"][0])
print("Advanced RAG 답변:", ans_adv)


정답: 대중교통체계
Advanced RAG 답변: 대중교통체계입니다.


## Step 5.5 : Self-RAG — 검색 필요성 판단 + 답변 자가 비평

Day2_1 노트에서 강조한 또 하나의 핵심 패턴, **Self-RAG** 입니다. Self-RAG의 핵심은 **LLM이 검색·답변 과정에 스스로 비평(critique)을 끼워 넣는다**는 점입니다.

이번 셀에서는 공식 Self-RAG 모델을 따로 받지 않고, **세 개의 작은 LLM 프롬프트**로 같은 흐름을 흉내내 봅니다.

1. **Retrieve 결정** — 질문이 들어오면, 외부 검색이 필요한지 LLM이 먼저 판단합니다. (`YES`/`NO` 한 단어)
2. **답변 생성** — `YES` 면 일반 RAG, `NO` 면 검색 없이 LLM 단독 답변.
3. **답변 자가 비평** — 생성된 답변이 컨텍스트에 충분히 근거하는지 LLM이 점검합니다. (`SUPPORTED` / `NOT_SUPPORTED`)
4. **보완 재시도** — `NOT_SUPPORTED` 면 Step 3의 **HyDE** 로 검색 쿼리를 바꿔 한 번 더 시도합니다.

코드 골격은 제공해 두었고, **두 군데 핵심 프롬프트만 여러분이 직접 채워주세요.**

In [11]:
# Self-RAG 아이디어를 프롬프트 기반으로 간소화한 구현입니다.
# 공식 Self-RAG 전용 모델의 reflection token을 학습한 구조는 아닙니다.

RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    """
질문에 외부 문서 검색이 필요한지 판단하세요.
특정 인물·사건·날짜·수치·문서 내용이 필요한 질문이면 YES,
간단한 계산이나 보편적인 정의만으로 충분하면 NO입니다.
오직 YES 또는 NO만 출력하세요.

질문: {question}
"""
)

CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    """
당신은 답변 근거 검사자입니다.

판정 기준:
- 답변이 짧은 단어나 구절이어도 문서에서 직접 확인되거나 같은 의미이면 SUPPORTED입니다.
- 답변에 문서에 없는 새로운 사실·숫자·날짜·인물·설명이 포함되면 NOT_SUPPORTED입니다.
- 질문이 요구한 대상과 답변이 일치하는지도 확인하세요.
- 오직 SUPPORTED 또는 NOT_SUPPORTED만 출력하세요.

[질문]
{question}

[문서]
{context}

[답변]
{answer}
"""
)

def normalize_label(text, allowed, default):
    text = str(text).strip().upper()
    for label in allowed:
        if label in text:
            return label
    return default


def self_rag(question, max_retries=1, verbose=True):
    decision_raw = (RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}
    )
    decision = normalize_label(decision_raw, ["YES", "NO"], "YES")
    if verbose:
        print(f"[1] Retrieve 필요? → {decision}")

    if decision == "NO":
        answer = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변")
        return answer, []

    docs = db.as_retriever(search_kwargs={"k": 5}).invoke(question)
    answer = "문서에서 확인할 수 없습니다."

    for attempt in range(max_retries + 1):
        top_docs = rerank(question, docs, top_k=3)
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({
            "context": format_docs(top_docs),
            "question": question,
        })
        critique_raw = (CRITIQUE_PROMPT | llm | StrOutputParser()).invoke({
            "question": question,
            "context": format_docs(top_docs),
            "answer": answer,
        })
        critique = normalize_label(
            critique_raw, ["NOT_SUPPORTED", "SUPPORTED"], "NOT_SUPPORTED"
        )
        if verbose:
            print(f"[3] 시도 {attempt + 1} — {critique}")

        if critique == "SUPPORTED":
            return answer, top_docs

        if attempt < max_retries and USE_HYDE_FALLBACK:
            hyde_docs, _ = hyde_retrieve(question, k=5)
            # 기존 검색과 HyDE 검색을 합치고 중복 제거한 뒤 재평가
            merged = {d.page_content: d for d in (docs + hyde_docs)}
            docs = list(merged.values())
            if verbose:
                print("[4] 근거 부족 → HyDE 재검색 후 후보 병합")

    return answer, top_docs

ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)


[1] Retrieve 필요? → YES
[3] 시도 1 — SUPPORTED

=== Self-RAG 최종 답변 ===
대중교통체계입니다.


## Step 6 : RAGAS 평가용 데이터셋 만들기

RAGAS 는 네 가지 자료가 필요합니다.
- `user_input` — 사용자 질문
- `response`   — RAG 가 생성한 답변
- `retrieved_contexts` — RAG 가 참고한 문서들
- `reference`  — 모범 답안 (Ground Truth)

**KorQuAD 는 사람이 작성한 정답이 데이터셋에 이미 포함**되어 있어, `reference` 를 따로 작성할 필요 없이 그대로 가져다 씁니다. 같은 질문 셋을 **Naive RAG** 와 **Advanced RAG** 두 가지로 풀고 결과를 비교합니다.

토큰 비용 통제를 위해 평가 질문은 5개만 사용합니다. (늘리려면 `EVAL_N` 변경)

In [12]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
eval_samples = list(raw_ds)[:EVAL_N]
questions = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

# 동일 세션에서 셀을 재실행해도 변수에 결과가 중복 누적되지 않도록 매번 초기화
naive_answers, naive_contexts = [], []
adv_answers, adv_contexts = [], []

for idx, q in enumerate(questions, 1):
    ctx = naive_retriever.invoke(q)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": format_docs(ctx), "question": q
    })
    naive_answers.append(answer)
    naive_contexts.append([d.page_content for d in ctx])

    adv_answer, adv_ctx = advanced_rag(q)
    adv_answers.append(adv_answer)
    adv_contexts.append([d.page_content for d in adv_ctx])
    print(f"[{idx}/{EVAL_N}] 답변 생성 완료")

print(f"평가 데이터 준비 완료 — {EVAL_N}개 질문 × 2개 파이프라인")


[1/8] 답변 생성 완료
[2/8] 답변 생성 완료
[3/8] 답변 생성 완료
[4/8] 답변 생성 완료
[5/8] 답변 생성 완료
[6/8] 답변 생성 완료
[7/8] 답변 생성 완료
[8/8] 답변 생성 완료
평가 데이터 준비 완료 — 8개 질문 × 2개 파이프라인


In [13]:
from datasets import Dataset

def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)

## Step 7 : RAGAS로 4대 지표 계산하기  

Judge LLM은 `gpt-4o-mini`로, 임베딩은 `text-embedding-3-small`로 설정합니다.  
(Judge에 더 강한 모델을 쓰면 채점은 더 정교해지지만 비용이 늘어납니다.)

In [14]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb = OpenAIEmbeddings(model="text-embedding-3-small")
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

if RUN_KORQUAD_RAGAS:
    print("=== Naive RAG 채점 ===")
    naive_result = evaluate(
        naive_ds,
        metrics=metrics,
        llm=judge_llm,
        embeddings=judge_emb,
        raise_exceptions=False,
    )

    print("=== Advanced RAG 채점 ===")
    adv_result = evaluate(
        adv_ds,
        metrics=metrics,
        llm=judge_llm,
        embeddings=judge_emb,
        raise_exceptions=False,
    )
else:
    print("RUN_KORQUAD_RAGAS=False — 비용 절약을 위해 평가를 건너뜁니다.")


=== Naive RAG 채점 ===


Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

=== Advanced RAG 채점 ===


Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

In [15]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

METRIC_COLS = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]

def summary(df, label):
    available = [c for c in METRIC_COLS if c in df.columns]
    avg = df[available].mean(numeric_only=True)
    avg.name = label
    return avg

def metric_meaning(metric):
    meanings = {
        "faithfulness": "답변의 내용이 검색 문서에 충실하게 근거한 정도",
        "answer_relevancy": "생성 답변이 질문에 직접적으로 대응한 정도",
        "context_precision": "검색된 상위 문서 중 실제 관련 문서가 차지하는 정도",
        "context_recall": "정답에 필요한 근거를 검색 문서가 포함한 정도",
    }
    return meanings.get(metric, metric)

if RUN_KORQUAD_RAGAS:
    naive_df = naive_result.to_pandas()
    adv_df = adv_result.to_pandas()

    compare = pd.concat([
        summary(naive_df, "Naive RAG"),
        summary(adv_df, "Advanced RAG"),
    ], axis=1)
    delta_korquad = compare["Advanced RAG"] - compare["Naive RAG"]

    print("=== KorQuAD 평균 지표 ===")
    display(compare.round(3))

    print("\n=== Delta: Advanced - Naive ===")
    display(delta_korquad.round(3).to_frame("Delta"))

    valid_delta = delta_korquad.dropna()
    if not valid_delta.empty:
        best_metric = valid_delta.idxmax()
        best_delta = valid_delta.max()
        print("\n=== 자동 결과 해석 ===")
        print(
            f"가장 크게 개선된 지표는 {best_metric}이며 "
            f"변화량은 {best_delta:+.3f}입니다."
        )
        print(f"이 지표는 '{metric_meaning(best_metric)}'를 의미합니다.")

        if "context_precision" in valid_delta:
            print(
                f"context_precision 변화량은 "
                f"{valid_delta['context_precision']:+.3f}입니다. "
                "이 값이 상승했다면 Cross-Encoder Reranker가 "
                "질문과 관련된 문서를 상위에 배치한 효과로 해석할 수 있습니다."
            )
        if "context_recall" in valid_delta:
            print(
                f"context_recall 변화량은 "
                f"{valid_delta['context_recall']:+.3f}입니다. "
                "이 값이 상승했다면 후보 검색 범위를 top-3에서 top-10으로 "
                "넓힌 덕분에 정답 근거가 후보 집합에 포함될 가능성이 높아진 것입니다."
            )
        if "faithfulness" in valid_delta:
            print(
                f"faithfulness 변화량은 "
                f"{valid_delta['faithfulness']:+.3f}입니다. "
                "상승했다면 더 적절한 근거 문서가 제공되어 답변의 문서 근거성이 "
                "향상된 것으로 볼 수 있습니다."
            )
        if "answer_relevancy" in valid_delta and valid_delta["answer_relevancy"] < 0:
            print(
                "answer_relevancy는 하락했습니다. KorQuAD 정답은 한 단어나 "
                "짧은 구절인 경우가 많아 RAGAS의 질문 역추론 방식에서 변동성이 "
                "커질 수 있으므로 실제 답변도 함께 확인해야 합니다."
            )
else:
    print("RUN_KORQUAD_RAGAS=False — 결과표와 자동 해석을 생성하지 않습니다.")


=== KorQuAD 평균 지표 ===


,Naive RAG,Advanced RAG
faithfulness,0.875,0.750
answer_relevancy,0.171,0.143
context_precision,0.917,1.000
context_recall,1.000,1.000



=== Delta: Advanced - Naive ===


,Delta
faithfulness,-0.125
answer_relevancy,-0.028
context_precision,0.083
context_recall,0.000



=== 자동 결과 해석 ===
가장 크게 개선된 지표는 context_precision이며 변화량은 +0.083입니다.
이 지표는 '검색된 상위 문서 중 실제 관련 문서가 차지하는 정도'를 의미합니다.
context_precision 변화량은 +0.083입니다. 이 값이 상승했다면 Cross-Encoder Reranker가 질문과 관련된 문서를 상위에 배치한 효과로 해석할 수 있습니다.
context_recall 변화량은 +0.000입니다. 이 값이 상승했다면 후보 검색 범위를 top-3에서 top-10으로 넓힌 덕분에 정답 근거가 후보 집합에 포함될 가능성이 높아진 것입니다.
faithfulness 변화량은 -0.125입니다. 상승했다면 더 적절한 근거 문서가 제공되어 답변의 문서 근거성이 향상된 것으로 볼 수 있습니다.
answer_relevancy는 하락했습니다. KorQuAD 정답은 한 단어나 짧은 구절인 경우가 많아 RAGAS의 질문 역추론 방식에서 변동성이 커질 수 있으므로 실제 답변도 함께 확인해야 합니다.


### 실험 결과 상세 분석 — KorQuAD

이번 실험은 동일한 8개 질문에 대해 **Naive RAG(top-3 검색)**와 **Advanced RAG(Dense top-10 후보 검색 → Cross-Encoder Reranking → top-3 선택)**를 비교하였다.

| 평가 지표 | Naive RAG | Advanced RAG | 변화량 |
|---|---:|---:|---:|
| Faithfulness | 0.875 | 0.750 | -0.125 |
| Answer Relevancy | 0.171 | 0.143 | -0.028 |
| Context Precision | 0.917 | 1.000 | +0.083 |
| Context Recall | 1.000 | 1.000 | 0.000 |

#### 1. Context Precision: 0.917 → 1.000

가장 뚜렷하게 개선된 지표는 `context_precision`으로, **0.083 상승하여 1.000**을 기록하였다. 이는 Advanced RAG가 최종적으로 LLM에 전달한 문서들이 질문과 매우 높은 관련성을 가졌다는 뜻이다.

Naive RAG는 벡터 유사도만으로 상위 3개 문서를 바로 선택하지만, Advanced RAG는 먼저 후보를 10개로 넓게 확보한 뒤 Cross-Encoder Reranker가 질문과 각 문서를 함께 읽고 관련성을 다시 계산한다. 그 결과 단순 임베딩 유사도만으로 상위에 포함될 수 있었던 불필요한 문서가 제거되고, 실제 정답 근거가 포함된 문서가 최종 top-3에 안정적으로 배치된 것으로 해석할 수 있다.

#### 2. Context Recall: 1.000 → 1.000

`context_recall`은 두 방식 모두 **1.000**으로 동일했다. Naive RAG도 이미 모든 평가 질문에서 정답에 필요한 근거를 검색 문서 안에 포함하고 있었기 때문에, 후보 수를 top-10으로 늘리더라도 추가적인 상승 여지가 없었다.

따라서 KorQuAD 실험에서 Advanced RAG의 핵심 효과는 “기존에 없던 정답 근거를 새로 찾았다”기보다, **이미 검색된 근거 중 더 관련성 높은 문서를 위쪽으로 정돈했다**는 데 있다. 즉 Recall보다 Precision 개선이 더 분명하게 나타난 실험이다.

#### 3. Faithfulness: 0.875 → 0.750

`faithfulness`는 **0.125 하락**하였다. 검색 문서의 정밀도는 높아졌지만, 생성 답변이 문서에 더 충실해지지는 않았다는 뜻이다.

이 결과는 검색 품질과 생성 품질이 항상 함께 움직이지 않는다는 점을 보여준다. Reranker가 적절한 문서를 선택했더라도 LLM이 답변을 생성하는 과정에서 문서에 없는 표현을 덧붙이거나, 여러 문서의 정보를 합치거나, 짧은 정답 대신 불필요한 설명을 생성하면 Faithfulness가 낮아질 수 있다. 또한 평가 문항이 8개뿐이므로 한두 문항의 생성 오류가 평균 점수에 크게 영향을 주었을 가능성도 있다.

따라서 Faithfulness를 높이기 위해서는 검색 단계뿐 아니라 다음과 같은 생성 단계의 보완이 필요하다.

- 정답을 한 단어나 짧은 구절로 제한
- 답변과 함께 근거 문장을 인용
- 문서에 없는 정보는 생성하지 않도록 프롬프트 강화
- 생성 후 근거 일치 여부를 다시 검사하는 Critique 단계 적용

#### 4. Answer Relevancy: 0.171 → 0.143

`answer_relevancy`는 **0.028 소폭 하락**하였다. 두 점수 모두 절대값이 낮은 편인데, 이는 KorQuAD 정답이 인물명·장소명·짧은 명사구처럼 매우 짧은 경우가 많기 때문이다.

RAGAS의 Answer Relevancy는 생성 답변을 이용해 원래 질문을 역으로 추론하고 의미적 유사성을 평가한다. 따라서 “서울”, “두 개”, 특정 인물명처럼 정답이 매우 짧으면 질문의 의미를 충분히 복원하기 어려워 점수가 낮거나 불안정하게 나타날 수 있다. 이 실험에서는 0.028의 차이가 작고 표본도 8개이므로, Advanced RAG가 실제로 질문 대응 능력을 의미 있게 악화시켰다고 단정하기는 어렵다.

#### 종합 해석

KorQuAD에서는 Advanced RAG가 **검색 결과의 정밀도는 분명히 높였지만, 생성 답변의 근거성과 질문 적합성까지 자동으로 향상시키지는 못했다.** 즉 Reranking은 검색 품질을 개선하는 데 효과적이었으나, 최종 성능을 높이려면 생성 프롬프트와 근거 검증 단계도 함께 개선해야 한다.

또한 본 정량 비교에 사용된 Advanced RAG는 **Dense top-10 검색 → Cross-Encoder Reranking → top-3 문서로 답변 생성** 구조이다. Multi-Query, RAG-Fusion, HyDE, Self-RAG는 별도로 구현한 실습이며, 이 비교 점수에 직접 포함된 것은 아니다.


### Quiz 답안

이번 KorQuAD 실험에서 가장 크게 개선된 지표는 **Context Precision**이다. Naive RAG의 0.917에서 Advanced RAG의 1.000으로 **0.083 상승**하였다.

이러한 향상은 Advanced RAG가 문서를 단순히 더 많이 검색했기 때문이 아니라, 먼저 top-10 후보를 확보한 뒤 다국어 Cross-Encoder Reranker가 질문과 문서의 관련성을 다시 평가하여 최종 top-3만 선택했기 때문이다. 이 과정을 통해 질문과 직접 관련되지 않은 문서가 제거되고, 정답 근거가 포함된 문서가 최종 컨텍스트에 더 정확하게 배치되었다.

반면 Context Recall은 두 방식 모두 1.000으로 동일했다. Naive RAG 단계에서도 이미 정답에 필요한 근거가 모두 검색되었으므로, 후보 수를 늘려도 더 이상 개선될 여지가 없었다. 따라서 이번 실험에서 Advanced RAG의 효과는 새로운 근거를 추가로 찾은 것보다 **검색 결과의 순도와 정렬 품질을 높인 것**으로 해석하는 것이 정확하다.

Faithfulness는 0.875에서 0.750으로 0.125 하락했고, Answer Relevancy도 0.171에서 0.143으로 0.028 하락하였다. 이는 검색 문서가 좋아졌다고 해서 최종 생성 답변까지 자동으로 좋아지는 것은 아니라는 점을 보여준다. LLM이 정답보다 긴 설명을 생성하거나 문서에 없는 표현을 덧붙이면 Faithfulness가 낮아질 수 있으며, KorQuAD처럼 정답이 매우 짧은 데이터에서는 Answer Relevancy 점수도 불안정하게 나타날 수 있다.

결론적으로 Advanced RAG는 **검색 문서 선별 정확도를 높이는 데는 효과적이었지만**, 생성 답변의 충실도와 직접성까지 개선하려면 근거 인용, 짧은 답변 강제, 생성 후 검증과 같은 추가 단계가 필요하다.


## Step 8 : (선택) 평가 데이터를 LLM으로 자동 생성하기  

현업에서는 모범 답안(`reference`)을 사람이 직접 작성하는 게 가장 큰 부담입니다.  
RAGAS는 **원본 문서만 주면 Question·Reference·Context 한 세트를 자동으로 만들어 주는** 기능을 제공합니다.  
자세한 사용법은 공식 문서를 참고하세요.

https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/

---
# 추가 실습 — KLUE-MRC 한국어 뉴스 MRC 벤치마크로 RAG 평가하기

메인 실습은 위키 기반 **KorQuAD v1** 으로 진행했습니다. 이번 추가 실습은 도메인을 바꿔, **한국어 뉴스 기사 기반의 MRC 벤치마크 KLUE-MRC** 위에서 같은 파이프라인을 처음부터 다시 조립해 봅니다.

**KLUE-MRC**
- 카카오·네이버 등 한국 NLP 팀이 함께 만든 한국어 표준 벤치마크 KLUE 의 MRC 태스크
- 한국어 **뉴스 기사** 기반 (KorQuAD 의 위키와 도메인이 다름)
- 사람이 직접 작성한 정답 포함
- **`is_impossible=True`** 인 답할 수 없는 질문도 일부 포함 → 데이터 필터링이 필요한 도전적 케이스

위키 기반 KorQuAD 와 비교했을 때 어떤 차이(질문 스타일, 검색 난이도, 점수 분포)가 나는지 직접 관찰해 보세요.

이번에도 일부만 샘플링해서 토큰 비용을 통제합니다.
- Vector DB 에 들어갈 unique context: 약 200개
- 평가 질문: 20개
- 예상 비용: GPT-4o-mini 기준 RAGAS 평가까지 합쳐서 약 \$0.10 ~ \$0.20

**📥 데이터셋 다운로드 / 출처**
- HuggingFace `datasets` 자동 다운로드: <https://huggingface.co/datasets/klue>
- KLUE 공식 사이트: <https://klue-benchmark.com/>
- KLUE 논문: <https://arxiv.org/abs/2105.09680>

> 다른 데이터셋으로 한 번 더 해보고 싶다면:  
> - MIRACL 한국어: <https://huggingface.co/datasets/miracl/miracl> (config: `ko`)  
> - 영어 SQuAD: <https://huggingface.co/datasets/rajpurkar/squad>

### Step A. 데이터셋 로드

`datasets` 라이브러리로 KLUE-MRC 를 한 줄에 받아옵니다. KLUE 는 여러 sub-task 가 있는 멀티태스크 벤치마크라서 config 이름 `"mrc"` 를 명시해야 합니다.

데이터셋 페이지: <https://huggingface.co/datasets/klue>

In [16]:
from datasets import load_dataset

ds_klue = load_dataset("klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
print({k: ds_klue[0][k] for k in ds_klue.column_names})

README.md: 0.00B [00:00, ?B/s]

mrc/train-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

mrc/validation-00000-of-00001.parquet:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17554 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5841 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
{'title': 'BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시', 'context': 'BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 

### Step B. Context 추출 + 중복 제거 (+ is_impossible 필터링)

KLUE-MRC 에는 KorQuAD 에는 없는 **`is_impossible=True`** 케이스가 섞여 있습니다 (= context 만 보고는 답할 수 없는 질문). 평가용 ground_truth 가 비어 있으면 RAGAS 의 `context_recall` 이 깨지므로, 답이 있는 샘플만 남기세요.

- `ds_klue.filter(lambda x: not x["is_impossible"])` 로 답 있는 것만 추리고
- `shuffle(seed=42).select(range(300))` 으로 300개 샘플링
- 그 중 `context` 필드 기준으로 중복 제거 (보통 150~200개)
- 각각을 `Document(page_content=..., metadata={"title": ex["title"]})` 로 감싸 `context_docs` 에 담기

In [17]:
from langchain_core.documents import Document

# 답할 수 없는 문항 제외 → 셔플 → 샘플링
KLUE_SAMPLE_N = 180 if QUICK_MODE else 300
answerable_klue = ds_klue.filter(lambda x: not x["is_impossible"])
klue_sampled = answerable_klue.shuffle(seed=42).select(
    range(min(KLUE_SAMPLE_N, len(answerable_klue)))
)

klue_unique = {}
for ex in klue_sampled:
    klue_unique.setdefault(ex["context"], ex["title"])

context_docs_klue = [
    Document(page_content=context, metadata={"title": title})
    for context, title in klue_unique.items()
]

print(f"answerable samples: {len(klue_sampled)}")
print(f"unique contexts: {len(context_docs_klue)}")
print("첫 context:", context_docs_klue[0].page_content[:200])


Filter:   0%|          | 0/5841 [00:00<?, ? examples/s]

answerable samples: 180
unique contexts: 180
첫 context: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step C. Embedding + VectorStore

메인 실습에서 만든 `embedding` (`OpenAIEmbeddings(model="text-embedding-3-small")`) 을 그대로 재사용해, `context_docs` 로 새 Chroma DB `db_klue` 를 만드세요. (메인 실습의 `db` 변수를 덮어쓰지 마세요. 비교가 안 됩니다.)

> ⚠️ **batch 적재 필수** — KLUE-MRC 의 뉴스 context 는 평균 토큰 수가 커서, 150개 이상을 한 번에 `Chroma.from_documents` 로 넘기면 OpenAI embeddings 의 **300k 토큰/요청 한도** 에 걸려 `BadRequestError` 가 납니다. 메인 cell 8 처럼 100개씩 batch 로 `add_documents` 호출하세요:
> ```python
> db_klue = Chroma(embedding_function=embedding)
> BATCH = 100
> for i in range(0, len(context_docs), BATCH):
>     db_klue.add_documents(context_docs[i:i+BATCH])
> ```

> 인덱싱 토큰 비용: 약 200개 context × 평균 600 토큰 ≈ **120k 토큰** (≈ \$0.003)

In [18]:
try:
    if "db_klue" in globals():
        db_klue.delete_collection()
except Exception:
    pass

db_klue = Chroma(
    collection_name="klue_mrc_rag_final",
    embedding_function=embedding,
)
BATCH = 50
for i in range(0, len(context_docs_klue), BATCH):
    db_klue.add_documents(context_docs_klue[i:i+BATCH])

print(f"db_klue 적재 완료 — {len(context_docs_klue)}개 context")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


db_klue 적재 완료 — 180개 context


### Step D. 평가용 질문/정답 세트 추출

Step B 에서 필터링·샘플링한 데이터 중 **앞에서 20개**를 평가용으로 떼어내세요.

- `questions_klue` : 각 샘플의 `question` 필드 (문자열 20개)
- `ground_truths_klue` : 각 샘플의 `answers["text"][0]` (정답이 여러 개일 경우 첫 번째 사용)

> 참고: KLUE-MRC 는 정답이 한 구절~한 문장 단위의 **extractive QA** 입니다. 짧은 정답은 RAGAS 의 `context_recall` 변동성을 키우는 경향이 있으니, 평균을 함께 봐주세요.

In [19]:
eval_samples_klue = list(klue_sampled)[:EVAL_N_KLUE]
questions_klue = [ex["question"] for ex in eval_samples_klue]
ground_truths_klue = [ex["answers"]["text"][0] for ex in eval_samples_klue]

print(f"questions_klue: {len(questions_klue)}개")
print("예시 질문:", questions_klue[0])
print("예시 정답:", ground_truths_klue[0])


questions_klue: 8개
예시 질문: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
예시 정답: 두 개


### Step E. Naive RAG 베이스라인 (KLUE)

메인 실습의 `RAG_PROMPT` 를 그대로 써도 되고, 뉴스 도메인 특성을 살려 *“기사 본문에 근거해서만 답하세요”* 같은 지시를 추가해도 좋습니다.

- `naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})`
- 체인 구조는 메인 Step 1 과 동일

In [20]:
KLUE_RAG_PROMPT = ChatPromptTemplate.from_template(
    """
다음 뉴스 기사만 근거로 질문에 답하세요.

규칙:
1. 질문에서 요구하는 대상과 단위를 정확히 확인하세요.
2. 기사에 여러 숫자가 있으면 질문과 직접 연결된 숫자만 선택하세요.
3. 정답은 기사에 실제로 등장하는 표현을 우선 사용하세요.
4. 답은 설명을 길게 덧붙이지 말고 한 구절 또는 한 문장으로 작성하세요.
5. 기사에서 확인할 수 없으면 '문서에서 확인할 수 없습니다.'라고 답하세요.

[뉴스 기사]
{context}

[질문]
{question}

[답변]
"""
)

naive_retriever_klue = db_klue.as_retriever(
    search_type="similarity", search_kwargs={"k": 3}
)
naive_chain_klue = (
    {"context": naive_retriever_klue | format_docs,
     "question": RunnablePassthrough()}
    | KLUE_RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("Q:", questions_klue[0])
print("정답:", ground_truths_klue[0])
print("Naive RAG:", naive_chain_klue.invoke(questions_klue[0]))


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: 국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
정답: 두 개
Naive RAG: 200여명의 계좌에서 3억원어치의 리플이 도난당했다.


### Step F. Multi-Query Retrieval

메인 Step 2 와 동일하게 `MultiQueryRetriever.from_llm(...)` 으로 KLUE 검색기를 감싸세요. 한국어 질문이 들어가면 gpt-4o-mini 가 한국어로 유사 질문을 만들어 줍니다.

확장 질문 로깅을 켜서 어떤 한국어 변형 질문이 만들어지는지 직접 눈으로 확인하세요.

In [21]:
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever_klue = MultiQueryRetriever.from_llm(
    retriever=db_klue.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

docs_mq_klue = multi_query_retriever_klue.invoke(questions_klue[0])
print(f"검색된 문서 수: {len(docs_mq_klue)}")

INFO:langchain.retrievers.multi_query:Generated queries: ['1. 국내에서 해킹으로 피해를 입은 리플이 포함된 통장의 수는 몇 개인가요?  ', '2. 한국에서 해킹 사건에 연루된 리플이 있는 통장 수는 얼마인가요?  ', '3. 국내에서 해킹으로 영향을 받은 리플이 포함된 계좌의 개수는 어떻게 되나요?']


검색된 문서 수: 6


### Step G. HyDE 직접 구현

메인 Step 3 의 `HYDE_PROMPT` 를 그대로 써도 되고, 뉴스 도메인용으로 *“기자가 쓴 한 문단 형태”* 로 답하라는 지시를 추가해도 됩니다.

`hyde_retrieve_klue(question, k=3)` 함수를 만들고 `db_klue` 위에서 동작하도록 하세요.

In [22]:
HYDE_PROMPT_KLUE = ChatPromptTemplate.from_template(
    """
다음 질문과 관련된 뉴스 기사에서 등장할 법한 핵심 개념과 표현을 포함한
검색용 가상 문단을 작성하세요.

중요 규칙:
- 최종 답변이 아니라 검색용 문단입니다.
- 확실하지 않은 인물명, 날짜, 금액, 계좌 수, 인원수 등 구체적 수치를 만들지 마세요.
- 질문의 핵심 대상과 단위가 드러나는 표현을 포함하세요.

질문: {question}

검색용 가상 뉴스 문단:
"""
)

hyde_generator_klue = HYDE_PROMPT_KLUE | llm | StrOutputParser()

def hyde_retrieve_klue(question, k=3):
    hypothetical = hyde_generator_klue.invoke({"question": question})
    docs = db_klue.similarity_search(hypothetical, k=k)
    return docs, hypothetical

docs_hyde_klue, hyp_klue = hyde_retrieve_klue(questions_klue[0])
print("가상 문단(HyDE):\n", hyp_klue[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde_klue))
print("첫 문서:", docs_hyde_klue[0].page_content[:200] if docs_hyde_klue else "검색 결과 없음")


가상 문단(HyDE):
 최근 국내에서 발생한 해킹 사건으로 인해 리플이 포함된 통장 계좌 수가 급증하고 있는 것으로 나타났다. 금융 당국은 이번 해킹으로 피해를 입은 계좌 수를 조사 중이며, 초기 보고에 따르면 다수의 개인 및 기업 계좌가 영향을 받았다고 전해졌다. 전문가들은 해킹 사건의 배후에 있는 사이버 범죄 조직의 정체와 함께, 리플과 같은 암호화폐의 안전성에 대한 우려가 커지고 있다고 경고하고 있다. 이에 따라 금융 기관들은 고객 보호를 위한 추가적인 보안 조치를 검토하고 있는 상황이다. 
---
검색된 문서 수: 3
첫 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step H. Multilingual Cross-encoder Reranker

메인 Step 4 에서 이미 `BAAI/bge-reranker-v2-m3` 같은 다국어 reranker 를 사용하고 있습니다. 추가 실습에서는:

- 메인의 `reranker` 인스턴스를 그대로 재사용하거나
- 다른 다국어 reranker 와 비교해 봐도 좋습니다:
  - `Alibaba-NLP/gte-multilingual-reranker-base` — <https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base>
  - `jinaai/jina-reranker-v2-base-multilingual` — <https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual>

`rerank_klue(query, docs, top_k=3)` 함수를 만드세요. (메인 Step 4 의 `rerank` 와 동일 구조)

In [23]:
reranker_klue = reranker

def rerank_klue(query, docs, top_k=3):
    if not docs:
        return []
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker_klue.predict(pairs, show_progress_bar=False)
    ranked = sorted(zip(docs, scores), key=lambda x: float(x[1]), reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates_klue = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(questions_klue[0])
top3_klue = rerank_klue(questions_klue[0], candidates_klue, top_k=3)
print(f"후보 {len(candidates_klue)}개 → Reranker 상위 {len(top3_klue)}개")
print("최상위 문서:", top3_klue[0].page_content[:200] if top3_klue else "검색 결과 없음")


후보 10개 → Reranker 상위 3개
최상위 문서: 국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step I. Advanced RAG 체인 (넓게 → Rerank → LLM)

메인 Step 5 흐름과 동일.
1. `db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)` 로 후보 10개
2. `rerank_klue(question, candidates, top_k=3)` 로 좁힘
3. `RAG_PROMPT` + `llm` 으로 답변 생성

함수가 `(answer, top_docs)` 둘 다 반환하도록 만들어 두면 다음 평가 단계에서 그대로 씁니다.

In [24]:
def advanced_rag_klue(question, candidate_k=10, final_k=3):
    candidates = db_klue.as_retriever(search_kwargs={"k": candidate_k}).invoke(question)
    top_docs = rerank_klue(question, candidates, top_k=final_k)

    if not top_docs:
        return "문서에서 확인할 수 없습니다.", []

    answer = (KLUE_RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": format_docs(top_docs),
        "question": question,
    })
    return answer, top_docs

ans_adv_klue, ctx_adv_klue = advanced_rag_klue(questions_klue[0])
print("정답:", ground_truths_klue[0])
print("Advanced RAG(KLUE):", ans_adv_klue)
print("정답 일치 여부는 문자열 일치만 보지 말고 의미와 단위를 함께 확인하세요.")


정답: 두 개
Advanced RAG(KLUE): 200여명의 계좌에서 3억원어치의 리플이 도난당했다.
정답 일치 여부는 문자열 일치만 보지 말고 의미와 단위를 함께 확인하세요.


### Step J. RAGAS 로 Naive vs Advanced 비교

메인 Step 6/7 흐름을 KLUE-MRC 변수(`_klue`) 로 옮겨 동일하게 수행하세요.

1. 20개 질문 각각을 Naive / Advanced 파이프라인에 돌려 답변과 컨텍스트 수집
2. `Dataset.from_dict({...})` 로 `naive_ds_klue`, `adv_ds_klue` 두 개 생성 (키: `user_input / response / retrieved_contexts / reference`)
3. `evaluate(..., metrics=[faithfulness, answer_relevancy, context_precision, context_recall], llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)` 두 번
4. 평균표로 비교

#### 결과 비교
KorQuAD와 KLUE-MRC의 결과를 함께 비교한 결과, KorQuAD는 위키 기반 설명형 문서로 의미 기반 검색의 효과가 크게 나타났으며 Context Precision 향상이 두드러졌다. 반면 KLUE-MRC는 뉴스 기사 특성상 숫자·날짜·인명 등이 함께 등장하여 검색 성능은 향상되었지만 생성 단계에서 필요한 정보를 정확히 선택하지 못하는 사례가 발생하였다. 따라서 뉴스 도메인에서는 Retrieval 성능뿐 아니라 Generation 단계의 정보 추출 능력을 함께 개선해야 함을 확인하였다.

In [25]:
naive_answers_klue, naive_contexts_klue = [], []
adv_answers_klue, adv_contexts_klue = [], []

for idx, q in enumerate(questions_klue, 1):
    ctx = naive_retriever_klue.invoke(q)
    naive_answer = (KLUE_RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": format_docs(ctx), "question": q
    })
    naive_answers_klue.append(naive_answer)
    naive_contexts_klue.append([d.page_content for d in ctx])

    adv_answer, adv_ctx = advanced_rag_klue(q)
    adv_answers_klue.append(adv_answer)
    adv_contexts_klue.append([d.page_content for d in adv_ctx])
    print(f"[{idx}/{EVAL_N_KLUE}] KLUE 답변 생성 완료")


def make_dataset_klue(answers, contexts):
    return Dataset.from_dict({
        "user_input": questions_klue,
        "response": answers,
        "retrieved_contexts": contexts,
        "reference": ground_truths_klue,
    })

naive_ds_klue = make_dataset_klue(naive_answers_klue, naive_contexts_klue)
adv_ds_klue = make_dataset_klue(adv_answers_klue, adv_contexts_klue)

if RUN_KLUE_RAGAS:
    print("=== Naive RAG(KLUE) 채점 ===")
    naive_result_klue = evaluate(
        naive_ds_klue, metrics=metrics,
        llm=judge_llm, embeddings=judge_emb,
        raise_exceptions=False,
    )

    print("=== Advanced RAG(KLUE) 채점 ===")
    adv_result_klue = evaluate(
        adv_ds_klue, metrics=metrics,
        llm=judge_llm, embeddings=judge_emb,
        raise_exceptions=False,
    )

    naive_df_klue = naive_result_klue.to_pandas()
    adv_df_klue = adv_result_klue.to_pandas()
    compare_klue = pd.concat([
        summary(naive_df_klue, "Naive RAG (KLUE)"),
        summary(adv_df_klue, "Advanced RAG (KLUE)"),
    ], axis=1)
    delta_klue = compare_klue["Advanced RAG (KLUE)"] - compare_klue["Naive RAG (KLUE)"]

    print(compare_klue.round(3))
    print("\nDelta (Advanced - Naive), KLUE-MRC:")
    print(delta_klue.round(3))

    if RUN_KORQUAD_RAGAS:
        print("\n=== KorQuAD Δ vs KLUE-MRC Δ ===")
        delta_compare = pd.concat(
            [delta_korquad, delta_klue],
            axis=1,
            keys=["KorQuAD Δ", "KLUE-MRC Δ"],
        )
        print(delta_compare.round(3))
else:
    print("RUN_KLUE_RAGAS=False — KLUE RAGAS 평가를 건너뜁니다.")


[1/8] KLUE 답변 생성 완료
[2/8] KLUE 답변 생성 완료
[3/8] KLUE 답변 생성 완료
[4/8] KLUE 답변 생성 완료
[5/8] KLUE 답변 생성 완료
[6/8] KLUE 답변 생성 완료
[7/8] KLUE 답변 생성 완료
[8/8] KLUE 답변 생성 완료
=== Naive RAG(KLUE) 채점 ===


Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

=== Advanced RAG(KLUE) 채점 ===


Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

                   Naive RAG (KLUE)  Advanced RAG (KLUE)
faithfulness                  0.750                0.625
answer_relevancy              0.235                0.235
context_precision             0.792                0.854
context_recall                0.875                1.000

Delta (Advanced - Naive), KLUE-MRC:
faithfulness        -0.125
answer_relevancy     0.000
context_precision    0.062
context_recall       0.125
dtype: float64

=== KorQuAD Δ vs KLUE-MRC Δ ===
                   KorQuAD Δ  KLUE-MRC Δ
faithfulness          -0.125      -0.125
answer_relevancy      -0.028       0.000
context_precision      0.083       0.062
context_recall         0.000       0.125


### Step K. (선택) 좀 더 큰 샘플로 통계적 신뢰도 확보

질문 20개로는 표본 분산이 커서 Naive vs Advanced 차이가 우연일 수도 있습니다. 토큰 비용이 허용된다면 50~100문항으로 늘려 paired t-test 같은 간단한 통계 검정으로 차이가 유의한지 확인해 보세요.

참고: `scipy.stats.ttest_rel(naive_df["faithfulness"], adv_df["faithfulness"])`

In [26]:
# 선택 실습: 기본값은 False이므로 실행 비용이 발생하지 않습니다.
from scipy.stats import ttest_rel

if RUN_BIG_STAT_TEST:
    EVAL_N_BIG = min(50, len(klue_sampled))
    eval_samples_big = list(klue_sampled)[:EVAL_N_BIG]
    questions_big = [ex["question"] for ex in eval_samples_big]
    ground_truths_big = [ex["answers"]["text"][0] for ex in eval_samples_big]

    naive_answers_big, naive_contexts_big = [], []
    adv_answers_big, adv_contexts_big = [], []

    for idx, q in enumerate(questions_big, 1):
        ctx = naive_retriever_klue.invoke(q)
        naive_answers_big.append(
            (KLUE_RAG_PROMPT | llm | StrOutputParser()).invoke({
                "context": format_docs(ctx), "question": q
            })
        )
        naive_contexts_big.append([d.page_content for d in ctx])

        adv_answer, adv_ctx = advanced_rag_klue(q)
        adv_answers_big.append(adv_answer)
        adv_contexts_big.append([d.page_content for d in adv_ctx])
        print(f"[{idx}/{EVAL_N_BIG}] 50문항 답변 생성")

    def make_dataset_big(answers, contexts):
        return Dataset.from_dict({
            "user_input": questions_big,
            "response": answers,
            "retrieved_contexts": contexts,
            "reference": ground_truths_big,
        })

    naive_df_big = evaluate(
        make_dataset_big(naive_answers_big, naive_contexts_big),
        metrics=metrics, llm=judge_llm, embeddings=judge_emb,
        raise_exceptions=False,
    ).to_pandas()
    adv_df_big = evaluate(
        make_dataset_big(adv_answers_big, adv_contexts_big),
        metrics=metrics, llm=judge_llm, embeddings=judge_emb,
        raise_exceptions=False,
    ).to_pandas()

    rows = []
    for col in METRIC_COLS:
        pair = pd.concat(
            [naive_df_big[col], adv_df_big[col]], axis=1
        ).dropna()
        pair.columns = ["naive", "advanced"]

        if len(pair) < 2:
            rows.append([col, len(pair), None, None, None, "표본 부족"])
            continue

        stat, pvalue = ttest_rel(pair["naive"], pair["advanced"])
        mean_delta = (pair["advanced"] - pair["naive"]).mean()
        conclusion = "유의함" if pvalue < 0.05 else "유의하지 않음"
        rows.append([col, len(pair), mean_delta, stat, pvalue, conclusion])

    stat_table = pd.DataFrame(
        rows,
        columns=["metric", "n", "mean_delta", "t_stat", "p_value", "conclusion"],
    )
    display(stat_table.round(4))
else:
    print("RUN_BIG_STAT_TEST=False — 50문항 통계 검정을 실행하지 않습니다.")


RUN_BIG_STAT_TEST=False — 50문항 통계 검정을 실행하지 않습니다.


### 마지막 Quiz 답안 및 KLUE-MRC 실험 결과 분석

#### 1. KorQuAD와 KLUE-MRC의 도메인 차이가 지표에 미치는 영향

KorQuAD는 위키백과 문단을 기반으로 하므로 설명형 문장이 많고, 하나의 문단 안에서 핵심 사실이 비교적 명확하게 정리되어 있다. 반면 KLUE-MRC는 뉴스 기사 기반이기 때문에 인물명, 기관명, 날짜, 금액, 계좌 수, 인용문처럼 서로 비슷한 정보가 한 기사 안에 동시에 등장한다.

실제 KLUE 예시에서 정답은 **“두 개”**였지만, Naive RAG와 Advanced RAG 모두 **“200여 명의 계좌에서 3억 원어치의 리플이 도난당했다”**고 답했다. 검색된 최상위 문서는 정답 기사를 정확히 포함하고 있었으므로, 이 오류는 검색 실패라기보다 **문서 내부에서 질문이 요구한 대상과 단위를 잘못 선택한 생성 오류**에 가깝다.

따라서 KLUE-MRC에서는 관련 기사를 찾는 능력뿐 아니라, 질문이 요구하는 개체와 단위를 정확히 식별하고 기사 안의 여러 숫자 중 올바른 숫자를 추출하는 능력이 매우 중요하다.

#### 2. KLUE-MRC의 정량 결과

| 평가 지표 | Naive RAG | Advanced RAG | 변화량 |
|---|---:|---:|---:|
| Faithfulness | 0.750 | 0.625 | -0.125 |
| Answer Relevancy | 0.235 | 0.235 | 0.000 |
| Context Precision | 0.792 | 0.854 | +0.062 |
| Context Recall | 0.875 | 1.000 | +0.125 |

가장 크게 개선된 지표는 `context_recall`로, **0.875에서 1.000으로 0.125 상승**하였다. Naive RAG의 top-3 검색에서는 일부 질문의 정답 근거가 누락되었지만, Advanced RAG가 후보 범위를 top-10으로 넓히면서 모든 평가 문항에서 정답 근거를 후보 집합에 포함시킨 것으로 해석할 수 있다.

`context_precision`도 **0.792에서 0.854로 0.062 상승**하였다. 후보 수를 늘리면 일반적으로 관련 없는 문서가 함께 증가할 수 있지만, Cross-Encoder Reranker가 관련 문서를 다시 선별했기 때문에 Recall과 Precision이 동시에 개선되었다.

반면 `faithfulness`는 **0.750에서 0.625로 0.125 하락**하였다. 정답 근거 문서를 더 잘 찾았음에도 최종 답변이 해당 근거를 정확히 사용하지 못한 경우가 있었다는 뜻이다. 앞서 확인한 리플 통장 개수 예시가 대표적이다. 정답 기사는 검색되었지만 LLM이 기사 안의 다른 숫자를 선택했다.

`answer_relevancy`는 두 방식 모두 **0.235**로 동일했다. 검색 방식이 바뀌어도 생성 답변의 질문 대응성이 개선되지 않았으며, 이는 검색기보다 최종 답변 생성 단계가 병목으로 남아 있음을 보여준다.

#### 3. KorQuAD와 KLUE-MRC 결과 비교

| 평가 지표 | KorQuAD 변화량 | KLUE-MRC 변화량 |
|---|---:|---:|
| Faithfulness | -0.125 | -0.125 |
| Answer Relevancy | -0.028 | 0.000 |
| Context Precision | +0.083 | +0.062 |
| Context Recall | 0.000 | +0.125 |

두 데이터셋 모두 Advanced RAG 적용 후 Context Precision이 상승하고 Faithfulness가 하락하였다. 이는 Reranker가 검색 문서를 정교하게 선별하는 데는 효과적이었지만, 더 좋은 컨텍스트가 반드시 더 충실한 답변으로 이어지지는 않았음을 의미한다.

차이점은 Context Recall이다. KorQuAD에서는 Naive RAG가 이미 1.000이었기 때문에 개선이 없었지만, KLUE-MRC에서는 0.875에서 1.000으로 상승했다. 뉴스 질문은 표현이 다양하고 기사 간 유사성이 높아 top-3만으로는 정답 근거를 놓칠 수 있으므로, top-10 후보 확장이 더 큰 효과를 보인 것으로 해석할 수 있다.

#### 4. KLUE-MRC에서 Advanced RAG의 효과

KLUE-MRC에서 Advanced RAG는 검색 단계에서는 분명한 효과를 보였다. Context Recall과 Context Precision이 모두 향상되었기 때문이다. 그러나 실제 답변 생성에서는 기사 안의 다른 숫자를 선택하는 오류가 남아 있었다.

따라서 뉴스 MRC에서는 다음 구조가 더 적합하다.

1. Dense Retrieval로 후보 기사 검색
2. Cross-Encoder Reranking으로 관련 기사 선별
3. 질문의 대상·단위·시간 표현 분석
4. 기사에서 근거 문장을 먼저 추출
5. 추출된 문장 안에서 짧은 정답만 생성
6. 답변과 근거 문장의 일치 여부 검증

즉 검색 품질 개선만으로는 충분하지 않으며, **Extractive QA 또는 근거 문장 추출 단계**가 추가되어야 한다.

#### 5. `is_impossible=True` 문항을 섞었을 때의 영향

`is_impossible=True` 문항은 제공된 문서에 정답 근거가 존재하지 않는다. 이러한 문항을 일반 문항과 섞으면 검색 시스템이 정답 근거를 회수할 수 없으므로 Context Recall이 낮아질 가능성이 크다.

또한 모델이 근거가 없는데도 임의로 답을 생성하면 Faithfulness도 하락한다. 반대로 “문서에서 확인할 수 없습니다”라고 적절히 응답하더라도, 일반적인 정답 기반 RAGAS 평가만으로는 정답 불가능 문항을 충분히 평가하기 어렵다.

따라서 `is_impossible=True` 문항은 별도로 분리하여 다음 항목을 평가하는 것이 적절하다.

- 정답 불가능 여부를 정확히 탐지했는가
- 근거 없는 답변 생성을 억제했는가
- 정보 부족 응답을 명확히 생성했는가
- 잘못된 확신을 표현하지 않았는가

#### 6. MIRACL 한국어 데이터로 이전했을 때 예상되는 차이

MIRACL은 다국어 검색 성능 평가를 목적으로 하므로 KorQuAD나 KLUE-MRC보다 검색기 자체의 차이가 더 크게 드러날 가능성이 있다. 질문과 문서 사이의 어휘 차이가 크거나, 고유명사·전문용어가 포함된 경우 Dense Retrieval만으로는 정확한 문서를 놓칠 수 있다.

따라서 MIRACL 한국어 환경에서는 다음 기법이 중요하다.

- Dense Retrieval과 BM25를 결합한 Hybrid Search
- 한국어 또는 다국어 특화 임베딩
- 다국어 Cross-Encoder Reranker
- Recall@k, MRR, nDCG 같은 검색 중심 지표
- RAGAS를 이용한 생성 품질 평가

검색 평가와 생성 평가를 분리하면 어느 단계에서 성능 손실이 발생하는지 더 명확하게 분석할 수 있다.


## 최종 결론

본 프로젝트에서는 KorQuAD v1(위키백과 기반) 과 KLUE-MRC(뉴스 기반) 데이터셋을 대상으로 Naive RAG와 Advanced RAG를 구현하고, RAGAS(Faithfulness, Answer Relevancy, Context Precision, Context Recall)를 이용하여 두 파이프라인의 성능을 정량적으로 비교하였다. 또한 Multi-Query Retrieval, RAG-Fusion(Reciprocal Rank Fusion 직접 구현), HyDE(Hypothetical Document Embedding), Cross-Encoder Reranking, Self-RAG 등 최근 RAG 시스템에서 활용되는 다양한 기법을 직접 구현하고 각 기법의 동작 원리와 효과를 확인하였다.

본 프로젝트에서 구현한 Advanced RAG는 Dense Retrieval(top-10) → Cross-Encoder Reranking → 상위 top-3 문서 선택 → LLM 답변 생성 구조를 사용하였다. 이와 별도로 Multi-Query, RAG-Fusion, HyDE, Self-RAG는 각 기법의 동작 원리와 검색 품질 향상 효과를 확인하기 위한 독립 실습으로 구현하였다. 따라서 RAGAS를 이용한 정량 비교 결과는 Dense Retrieval과 Cross-Encoder Reranking을 적용한 Advanced RAG의 효과를 중심으로 해석하였다.

KorQuAD 데이터셋에서는 Advanced RAG 적용 후 Context Precision이 향상되었으며, Context Recall은 Naive RAG에서도 이미 1.000으로 나타났다. 이는 필요한 문서를 새롭게 더 많이 검색한 것이 아니라, 기존 검색 후보 중 질문과 가장 관련성이 높은 문서를 상위에 배치함으로써 검색 품질이 개선되었음을 의미한다. 즉, Cross-Encoder Reranking이 문서의 의미적 관련성을 효과적으로 반영하여 Retrieval 단계의 정확도를 높인 것으로 해석할 수 있다.

KLUE-MRC 데이터셋에서는 Context Precision이 0.792에서 0.854로 증가(+0.062) 하였고, Context Recall은 0.875에서 1.000으로 증가(+0.125) 하였다. 이는 검색 후보를 Top-10까지 확장한 후 Cross-Encoder Reranking을 적용함으로써 정답이 포함된 문서를 상위에 배치하는 능력이 향상되었음을 보여준다. 특히 뉴스 기사에는 다양한 날짜, 숫자, 기관명, 인물명이 함께 등장하기 때문에 단순 Dense Retrieval보다 Reranking의 효과가 더욱 크게 나타난 것으로 판단된다.

반면 Faithfulness는 KorQuAD와 KLUE-MRC 모두에서 0.125 감소하였다. 또한 Answer Relevancy는 KorQuAD에서는 소폭 감소하였고, KLUE-MRC에서는 동일한 값을 유지하였다. 이러한 결과는 Retrieval 성능이 향상되더라도 생성 모델이 검색된 근거를 정확하게 활용하지 못하면 최종 답변의 품질은 반드시 함께 향상되지 않는다는 점을 보여준다.

실제 KLUE-MRC 실험에서는 정답 문서가 검색되었음에도 불구하고 질문의 정답인 "두 개" 대신 기사에 함께 등장한 "200여 명의 계좌" 를 답변으로 생성하는 사례가 확인되었다. 이는 Retrieval 단계는 성공했지만 Generation 단계에서 문서 내 여러 숫자와 개체명 가운데 질문과 직접 관련된 정보를 올바르게 선택하지 못한 사례로 해석할 수 있다. 따라서 Retrieval 성능 향상만으로는 최종 답변 품질을 보장할 수 없으며, 생성 모델의 정보 선택 능력과 프롬프트 설계 또한 매우 중요한 요소임을 확인하였다.

이번 프로젝트를 통해 Retrieval과 Generation은 서로 다른 최적화 대상이라는 점을 확인할 수 있었다. Retrieval 단계에서는 Context Precision과 Context Recall을 높여 필요한 근거 문서를 정확하게 확보하는 것이 중요하며, Generation 단계에서는 확보된 문서에서 질문과 직접 관련된 정보만을 선택하여 답변을 생성하는 능력이 중요하다. 따라서 실제 서비스 수준의 RAG 시스템에서는 검색, 재정렬, 생성, 검증 단계를 각각 독립적으로 개선하는 것이 아니라 하나의 통합된 파이프라인으로 함께 설계해야 안정적인 성능을 기대할 수 있다.

또한 Multi-Query Retrieval은 하나의 질문을 여러 개의 검색 질의로 확장하여 검색 다양성을 높였고, RAG-Fusion은 여러 검색 결과를 RRF 알고리즘으로 통합하여 검색 안정성을 향상시켰다. HyDE는 검색에 적합한 가상의 문서를 생성하여 의미 기반 검색 성능을 높였으며, Cross-Encoder Reranking은 질문과 문서의 의미적 관련성을 다시 계산하여 검색 정확도를 향상시켰다. Self-RAG는 검색 필요 여부를 판단하고 생성된 답변을 다시 검토하는 과정을 추가하여 답변의 신뢰성을 높이고자 하였다. 이처럼 각 기법은 서로 다른 역할을 수행하며, 실제 RAG 시스템에서는 목적에 따라 적절히 조합하여 사용하는 것이 중요함을 확인하였다.

이번 실험에는 몇 가지 한계도 존재하였다. 비용과 실행 시간을 고려하여 QUICK 모드를 사용함에 따라 데이터셋당 8개 문항만 평가하였기 때문에 표본 수가 충분하지 않았으며, 통계적 변동성이 존재할 가능성이 있다. 또한 RAGAS는 LLM 기반 평가 방식이므로 동일한 코드를 실행하더라도 실행 시점에 따라 일부 점수가 달라질 수 있다. 통계적 유의성을 확인하기 위한 paired t-test 역시 비용 절감을 위해 실제 대규모 실험은 수행하지 못하였다. 또한 본 프로젝트의 Self-RAG는 공식 Self-RAG 논문의 Reflection Token 기반 모델이 아니라 프롬프트 기반으로 구현한 간소화 버전이라는 한계가 있다.

향후에는 BM25와 Dense Retrieval을 결합한 Hybrid Search, Query Rewrite, Query Routing, Context Compression, 근거 문장 추출 기반 Extractive QA, Answer Verification 등을 추가 적용하여 Retrieval과 Generation을 함께 개선할 계획이다. 또한 더 많은 평가 문항을 이용하여 Recall@k, MRR, nDCG 등 검색 전용 지표를 함께 분석한다면 보다 신뢰성 있는 성능 평가가 가능할 것으로 기대된다.

이번 프로젝트를 통해 RAG 시스템의 성능은 단순히 더 많은 문서를 검색하는 것으로 결정되는 것이 아니라, Retrieval, Reranking, Generation, Verification이 하나의 유기적인 파이프라인으로 함께 동작할 때 더욱 안정적이고 신뢰성 높은 답변을 생성할 수 있음을 확인하였다. 또한 데이터의 특성에 따라 적절한 검색 전략과 생성 방식이 달라질 수 있으며, 실제 서비스 수준의 RAG 시스템에서는 검색 성능 향상뿐 아니라 생성 모델의 근거 활용 능력과 답변 검증 과정까지 함께 최적화하는 것이 중요하다는 점을 확인하였다.
